# Jour 3 — Distillation de connaissances

Le teacher (embeddings Evo2 + MLP) est précis mais dépend du fait qu'un modèle à 7
milliards de paramètres ait déjà traité chaque entrée. Peut-on compresser ce qu'il a
appris dans quelque chose de minuscule qui ne regarde que des comptages de k-mers ?

**Distillation** : entraîner un petit « student » non seulement sur les étiquettes dures
0/1, mais sur les prédictions *douces* du teacher (sa probabilité, adoucie par une
température) — l'incertitude du teacher porte une information (« *dark knowledge* ») que
les étiquettes dures ne portent pas.

In [ ]:
import sys
sys.path.append("src")

import torch
from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix
from models.classifier_heads import MLPHead
from models.distillation import train_student, distillation_loss
from eval import evaluate_logits, count_params, measure_latency_torch

EMB_DIR = "../../data/supervised/embeddings"
PROCESSED_DIR = "../../data/supervised/processed"

# on recharge les données d'entraînement du teacher + un teacher neuf (ou réutilisez celui du notebook 02)
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")

# TODO : instanciez un nouveau MLPHead(d_in=X_train_emb.shape[1])
teacher = ...
# NOTE : si vous avez sauvegardé les poids du teacher au notebook 02, chargez-les ici plutôt que de le ré-entraîner.

## Faire correspondre les entrées du student aux fenêtres exactes notées par le teacher

Les embeddings ont été extraits pour un sous-échantillon de fenêtres (voir
`extract_evo2_embeddings.py --max_per_split`), identifiées par `ids`. Il nous faut les
séquences brutes de ces *mêmes* fenêtres pour calculer les caractéristiques k-mer du
student.

In [ ]:
splits = load_all(PROCESSED_DIR)
train_df = splits["train"].set_index("id")
val_df = splits["val"].set_index("id")

# TODO : récupérez les séquences correspondant à ids_train / ids_val (train_df.loc[..., "sequence"])
train_seqs = ...
val_seqs = ...

# TODO : calculez les matrices de k-mer (k=4) pour ces séquences, convertissez en tensors torch
X_train_kmer = ...
X_val_kmer = ...
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_train_emb_t = torch.tensor(X_train_emb, dtype=torch.float32)

In [ ]:
# TODO : (ré-)entraînez rapidement le teacher sur cet échantillon exact (20 époques,
# Adam lr=1e-3, mini-lots de 256, BCE-with-logits), puis passez-le en mode eval et
# calculez ses logits sur train
optimizer = ...
n = X_train_emb_t.shape[0]
for epoch in range(20):
    perm = torch.randperm(n)
    for start in range(0, n, 256):
        idx = perm[start:start + 256]
        optimizer.zero_grad()
        loss = ...
        loss.backward()
        optimizer.step()

teacher.eval()
with torch.no_grad():
    teacher_logits_train = ...

## Entraînons le student avec la perte hybride (entropie croisée + distillation par cibles douces)

In [ ]:
# TODO : instanciez un student MLPHead volontairement minuscule (d_in=X_train_kmer.shape[1], d_hidden=32)
student = ...

# TODO : entraînez-le avec train_student(...) — 30 époques, temperature=4.0, alpha=0.5
student, history = ...

In [ ]:
# TODO : évaluez le student sur validation, puis calculez metrics/params/latence
student.eval()
with torch.no_grad():
    val_logits = ...
student_metrics = ...
print("distilled student:", student_metrics)
print("params:", ..., "| latency (ms/sample):", ...)

### Point de contrôle

Le student devrait récupérer la majeure partie de la précision du teacher tout en étant
bien plus petit (aucune dépendance à Evo2 à l'inférence — seulement des comptages de
k-mers + un petit MLP) et bien plus rapide. Essayez de faire varier `temperature` et
`alpha` dans `train_student` pour observer le compromis.

Suite : `04_compression_analysis_and_wrapup.ipynb` — mettre tous les modèles sur un seul
graphique.

*Bloqué ? La version complète est dans `solution/03_knowledge_distillation.ipynb`.*